# mT5-Gold Seq2Seq RAG

This notebook implements **Gold-context fine-tuning** for `google/mt5-base` and then uses the same saved generator in three RAG configurations:

1. **TF-IDF + mT5-Gold**
2. **BM25 + mT5-Gold**
3. **Dense + mT5-Gold**

## Experimental design

### Training
The model is trained exclusively on:

`processed_question + gold context -> processed_answer`

The gold context is built only from chunks that cover the annotated `source_pages`.

### Validation
The validation set is **not used to learn parameters**. It is used for:
- monitoring `eval_loss`,
- early stopping,
- selecting the best checkpoint,
- later end-to-end comparison of the three RAG configurations.

### Test
The test set is not used for tuning or model selection. Test inference is intentionally disabled by default in this notebook (`RUN_FINAL_TEST = False`) and is run only after the model and RAG configuration are finalized.

### Important notes
- The generator uses `processed_text` / `processed_question`, not lexical representation
- The model, tokenizer, configuration, training history, and generated predictions are saved

In [1]:
!git clone https://github.com/anjaanjaa10/Student-Question-Answering-from-Course-Materials.git

%cd /content/Student-Question-Answering-from-Course-Materials

Cloning into 'Student-Question-Answering-from-Course-Materials'...
remote: Enumerating objects: 674, done.
remote: Counting objects: 100% (674/674), done.
remote: Compressing objects: 100% (491/491), done.
remote: Total 674 (delta 309), reused 524 (delta 165), pack-reused 0 (from 0)
Receiving objects: 100% (674/674), 7.53 MiB | 13.97 MiB/s, done.
Resolving deltas: 100% (309/309), done.
/content/Student-Question-Answering-from-Course-Materials


In [3]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
import json
import math
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    set_seed,
)

### Changes from the first run

- `MAX_INPUT_LENGTH`: **512 → 1024**
- context is packed to a token budget instead of being blindly truncated
- `NUM_TRAIN_EPOCHS`: **5 → 8**
- early-stopping patience: **1 → 2**
- explicit `max_grad_norm=1.0`
- this run uses a separate artifact folder: `mt5_gold_1024`

In [5]:
SEED = 42

EXPERIMENT_NAME = "mt5_gold_v2_fp32"
MODEL_NAME = "google/mt5-base"

# MAX_INPUT_LENGTH = 512
MAX_INPUT_LENGTH = 1024
MAX_TARGET_LENGTH = 192

# Training gold context i RAG inference koriste isti maximanlni broj context chunks da smanje mismatch izmedju treninga i inferencije
RAG_TOP_K = 5
MAX_GOLD_CONTEXT_CHUNKS = RAG_TOP_K

LEARNING_RATE = 1e-4
NUM_TRAIN_EPOCHS = 8
TRAIN_BATCH_SIZE = 1
EVAL_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
WEIGHT_DECAY = 0.01
WARMUP_FRACTION = 0.10
# EARLY_STOPPING_PATIENCE = 1
EARLY_STOPPING_PATIENCE = 2
MAX_GRAD_NORM = 1.0

GENERATION_MAX_NEW_TOKENS = 192
GENERATION_NUM_BEAMS = 4
GENERATION_BATCH_SIZE = 1

TOP_K_EXPORTED = 10

set_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

USE_FP16 = False
USE_BF16 = False


In [6]:
PROJECT_ROOT = Path(
    "/content/Student-Question-Answering-from-Course-Materials"
)

if not (PROJECT_ROOT / "data").exists():
    raise FileNotFoundError(
        "Pokreni notebook iz root direktorijuma projekta "
        "(direktorijuma koji sadrži folder 'data')."
    )

CHUNKS_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "chunks_preprocessed.jsonl"
)

TRAIN_PATH = PROJECT_ROOT / "data" / "splits" / "train.jsonl"
VALIDATION_PATH = PROJECT_ROOT / "data" / "splits" / "validation.jsonl"
TEST_PATH = PROJECT_ROOT / "data" / "splits" / "test.jsonl"

RETRIEVAL_DIR = PROJECT_ROOT / "data" / "retrieval"

# ARTIFACTS_DIR = Path(
#     "/content/drive/MyDrive/Student-QA/artifacts/seq2seq/mt5_gold"
# )
ARTIFACTS_DIR = Path(
    "/content/drive/MyDrive/Student-QA/artifacts/seq2seq/mt5_gold_v2_fp32"
)

CHECKPOINT_DIR = ARTIFACTS_DIR / "checkpoints"
BEST_MODEL_DIR = ARTIFACTS_DIR / "best_model"
PREPARED_DATA_DIR = ARTIFACTS_DIR / "prepared_data"

GENERATION_DIR = Path(
    "/content/drive/MyDrive/Student-QA/artifacts/seq2seq/"
    "mt5_gold_v2_fp32/generation"
)

for directory in [
    ARTIFACTS_DIR,
    CHECKPOINT_DIR,
    BEST_MODEL_DIR,
    PREPARED_DATA_DIR,
    GENERATION_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

LOG_PATH = ARTIFACTS_DIR / "training.log"

logger = logging.getLogger("mt5_gold")
logger.setLevel(logging.INFO)
logger.handlers.clear()

file_handler = logging.FileHandler(
    LOG_PATH,
    mode="a",
    encoding="utf-8"
)
file_handler.setFormatter(
    logging.Formatter(
        "%(asctime)s | %(levelname)s | %(message)s"
    )
)
logger.addHandler(file_handler)

logger.info("Notebook started.")
logger.info("Device: %s", DEVICE)
logger.info("Model: %s", MODEL_NAME)

INFO:mt5_gold:Notebook started.
INFO:mt5_gold:Device: cuda
INFO:mt5_gold:Model: google/mt5-base


In [7]:
def load_jsonl(path: Path) -> list[dict]:
    with path.open("r", encoding="utf-8") as file:
        return [
            json.loads(line)
            for line in file
            if line.strip()
        ]


def save_jsonl(records: list[dict], path: Path) -> None:
    with path.open("w", encoding="utf-8") as file:
        for record in records:
            file.write(
                json.dumps(
                    record,
                    ensure_ascii=False
                )
                + "\n"
            )


for required_path in [
    CHUNKS_PATH,
    TRAIN_PATH,
    VALIDATION_PATH,
    TEST_PATH,
]:
    if not required_path.exists():
        raise FileNotFoundError(required_path)


chunks = load_jsonl(CHUNKS_PATH)
train_data = load_jsonl(TRAIN_PATH)
validation_data = load_jsonl(VALIDATION_PATH)
test_data = load_jsonl(TEST_PATH)

logger.info("Chunkovi: %d", len(chunks))
logger.info("Train pitanja: %d", len(train_data))
logger.info("Validation pitanja: %d", len(validation_data))
logger.info("Test pitanja: %d", len(test_data))

# print(chunks[0])
# print(train_data[0])

INFO:mt5_gold:Chunkovi: 356
INFO:mt5_gold:Train pitanja: 100
INFO:mt5_gold:Validation pitanja: 21
INFO:mt5_gold:Test pitanja: 22


### Build gold context and training/validation examples

The gold context is selected only through `source_pages`.

If multiple chunks overlap the gold pages:
1. chunks with a larger number of overlapping gold pages have priority,
2. at most `MAX_GOLD_CONTEXT_CHUNKS` chunks are selected,
3. the selected chunks are then returned to the document's original order.

This ensures that retrieval scores are not used during gold training.

In [8]:
# Load tokenizer before building training examples because the context packer
# needs the real mT5 token budget
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=False,
    legacy=True,
)

SYSTEM_PROMPT = (
    "Odgovori na pitanje isključivo na osnovu datog konteksta. "
    "Ako odgovor nije sadržan u kontekstu, reci da nemaš dovoljno informacija. "
    "Odgovori jasno, sažeto i na srpskom jeziku."
)

config.json:   0%|          | 0.00/702 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/376 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

In [9]:
sentinel_token_ids = [
    tokenizer.convert_tokens_to_ids(f"<extra_id_{i}>")
    for i in range(100)
]

sentinel_token_ids = [
    token_id
    for token_id in sentinel_token_ids
    if token_id is not None
]

In [10]:
def chunk_pages(chunk: dict) -> set[int]:
    return set(
        range(
            int(chunk["pdf_page_start"]),
            int(chunk["pdf_page_end"]) + 1
        )
    )


def get_gold_chunks(
    source_pages,
    max_chunks: int = MAX_GOLD_CONTEXT_CHUNKS
) -> list[dict]:

    if isinstance(source_pages, int):
        source_pages = [source_pages]

    source_pages = set(source_pages)
    candidates = []

    for corpus_index, chunk in enumerate(chunks):
        overlap = len(
            chunk_pages(chunk) & source_pages
        )

        if overlap > 0:
            candidates.append({
                "overlap": overlap,
                "corpus_index": corpus_index,
                "chunk": chunk,
            })

    # Prefer chunks covering more gold pages.
    candidates.sort(
        key=lambda item: (
            -item["overlap"],
            item["corpus_index"]
        )
    )

    selected = candidates[:max_chunks]

    # Restore document order for the generator.
    selected.sort(
        key=lambda item: item["corpus_index"]
    )

    return [
        item["chunk"]
        for item in selected
    ]


def build_model_input(
    processed_question: str,
    context: str
) -> str:
    return (
        f"{SYSTEM_PROMPT}\n\n"
        f"Pitanje: {processed_question}\n"
        f"Kontekst: {context}\n"
        f"Odgovor:"
    )


def token_length(text: str) -> int:
    return len(
        tokenizer(
            text,
            truncation=False,
            add_special_tokens=True
        )["input_ids"]
    )


def pack_context_to_input_budget(
    processed_question: str,
    selected_chunks: list[dict],
    max_input_length: int = MAX_INPUT_LENGTH,
) -> tuple[str, int, bool]:
    #Pack ranked/selected chunks into the mT5 input budget

    separator = "\n\n---\n\n"
    accepted_parts = []
    represented_chunks = 0
    shortened_last_chunk = False

    for chunk in selected_chunks:
        text = chunk.get("processed_text", "").strip()
        if not text:
            continue

        candidate_parts = accepted_parts + [text]
        candidate_context = separator.join(candidate_parts)

        if token_length(
            build_model_input(
                processed_question,
                candidate_context
            )
        ) <= max_input_length:
            accepted_parts.append(text)
            represented_chunks += 1
            continue

        # The whole next chunk does not fit. Keep the largest prefix that does.
        chunk_token_ids = tokenizer(
            text,
            truncation=False,
            add_special_tokens=False
        )["input_ids"]

        low = 0
        high = len(chunk_token_ids)
        best_prefix = ""

        while low <= high:
            mid = (low + high) // 2

            prefix = tokenizer.decode(
                chunk_token_ids[:mid],
                skip_special_tokens=True
            ).strip()

            partial_parts = (
                accepted_parts + [prefix]
                if prefix
                else accepted_parts
            )

            partial_context = separator.join(
                partial_parts
            )

            current_length = token_length(
                build_model_input(
                    processed_question,
                    partial_context
                )
            )

            if current_length <= max_input_length:
                best_prefix = prefix
                low = mid + 1
            else:
                high = mid - 1

        if best_prefix:
            accepted_parts.append(best_prefix)
            represented_chunks += 1
            shortened_last_chunk = True

        break

    context = separator.join(accepted_parts)

    return (
        context,
        represented_chunks,
        shortened_last_chunk
    )


In [11]:
def prepare_gold_examples(
    data: list[dict],
    split_name: str
) -> list[dict]:

    prepared = []
    missing_context_ids = []

    for example in data:
        gold_chunks = get_gold_chunks(
            example["source_pages"]
        )

        (
            context,
            n_context_chunks_used,
            context_shortened,
        ) = pack_context_to_input_budget(
            example["processed_question"],
            gold_chunks,
        )

        if not context.strip():
            missing_context_ids.append(
                example["id"]
            )
            continue

        target = example.get(
            "processed_answer",
            example.get("answer", "")
        ).strip()

        if not target:
            raise ValueError(
                f"Nedostaje target odgovor za ID "
                f"{example['id']} u splitu {split_name}."
            )

        model_input = build_model_input(
            example["processed_question"],
            context
        )

        if token_length(model_input) > MAX_INPUT_LENGTH:
            raise RuntimeError(
                f"Token-budget greška za ID {example['id']}: "
                f"input je i dalje duži od {MAX_INPUT_LENGTH}."
            )

        prepared.append({
            "question_id": example["id"],
            "input_text": model_input,
            "target_text": target,
            "source_pages": example["source_pages"],
            "n_gold_chunks_available": len(gold_chunks),
            "n_context_chunks_used": n_context_chunks_used,
            "context_shortened": context_shortened,
        })

    if missing_context_ids:
        raise ValueError(
            f"Gold context nedostaje za {split_name} IDs: "
            f"{missing_context_ids}. "
            "Proveri corpus/source_pages pre treninga."
        )

    logger.info(
        "%s prepared examples: %d",
        split_name,
        len(prepared)
    )

    return prepared


train_gold = prepare_gold_examples(
    train_data,
    "train"
)

validation_gold = prepare_gold_examples(
    validation_data,
    "validation"
)

# Test is intentionally not used for training/model selection.

save_jsonl(
    train_gold,
    PREPARED_DATA_DIR / "train_gold_v2_fp32.jsonl"
)

save_jsonl(
    validation_gold,
    PREPARED_DATA_DIR / "validation_gold_v2_fp32.jsonl"
)

INFO:mt5_gold:train prepared examples: 100
INFO:mt5_gold:validation prepared examples: 21


### Load the tokenizer and inspect input lengths

Length statistics are saved to `input_length_stats.json`

In [12]:
def sequence_lengths(
    records: list[dict],
    field: str
) -> list[int]:
    return [
        token_length(record[field])
        for record in records
    ]


train_input_lengths = sequence_lengths(
    train_gold,
    "input_text"
)

validation_input_lengths = sequence_lengths(
    validation_gold,
    "input_text"
)

train_target_lengths = sequence_lengths(
    train_gold,
    "target_text"
)

validation_target_lengths = sequence_lengths(
    validation_gold,
    "target_text"
)


def split_length_stats(
    records,
    input_lengths,
    target_lengths
):
    return {
        "n": len(records),
        "input_mean": float(np.mean(input_lengths)),
        "input_median": float(np.median(input_lengths)),
        "input_max": int(max(input_lengths)),
        "input_over_limit": int(
            sum(
                length > MAX_INPUT_LENGTH
                for length in input_lengths
            )
        ),
        "target_max": int(max(target_lengths)),
        "target_over_limit": int(
            sum(
                length > MAX_TARGET_LENGTH
                for length in target_lengths
            )
        ),
        "contexts_shortened": int(
            sum(
                bool(record["context_shortened"])
                for record in records
            )
        ),
        "mean_context_chunks_used": float(
            np.mean([
                record["n_context_chunks_used"]
                for record in records
            ])
        ),
    }


length_stats = {
    "max_input_length": MAX_INPUT_LENGTH,
    "max_target_length": MAX_TARGET_LENGTH,
    "train": split_length_stats(
        train_gold,
        train_input_lengths,
        train_target_lengths,
    ),
    "validation": split_length_stats(
        validation_gold,
        validation_input_lengths,
        validation_target_lengths,
    ),
}

with (ARTIFACTS_DIR / "input_length_stats.json").open(
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        length_stats,
        file,
        ensure_ascii=False,
        indent=2
    )

logger.info(
    "Train inputs over %d tokens: %d/%d",
    MAX_INPUT_LENGTH,
    length_stats["train"]["input_over_limit"],
    length_stats["train"]["n"]
)

logger.info(
    "Validation inputs over %d tokens: %d/%d",
    MAX_INPUT_LENGTH,
    length_stats["validation"]["input_over_limit"],
    length_stats["validation"]["n"]
)


# print(json.dumps(length_stats, indent=2, ensure_ascii=False))

INFO:mt5_gold:Train inputs over 1024 tokens: 0/100
INFO:mt5_gold:Validation inputs over 1024 tokens: 0/21


### Tokenize datasets

In [13]:
train_dataset = Dataset.from_list(
    train_gold
)

validation_dataset = Dataset.from_list(
    validation_gold
)


def tokenize_batch(batch):
    model_inputs = tokenizer(
        batch["input_text"],
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
    )

    labels = tokenizer(
        text_target=batch["target_text"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs


tokenized_train = train_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=train_dataset.column_names
)

tokenized_validation = validation_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=validation_dataset.column_names
)


Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/21 [00:00<?, ? examples/s]

### Model and training configuration

The best checkpoint is selected based on **validation `eval_loss`**

In [14]:
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
)

# Smanjuje memorijsku potrošnju tokom fine-tuninga.
model.config.use_cache = False

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
)

optimizer_steps_per_epoch = math.ceil(
    len(tokenized_train)
    / (
        TRAIN_BATCH_SIZE
        * GRADIENT_ACCUMULATION_STEPS
    )
)

TOTAL_OPTIMIZER_STEPS = (
    optimizer_steps_per_epoch
    * NUM_TRAIN_EPOCHS
)

WARMUP_STEPS = max(
    1,
    round(
        WARMUP_FRACTION
        * TOTAL_OPTIMIZER_STEPS
    )
)


training_args = Seq2SeqTrainingArguments(
    output_dir=str(CHECKPOINT_DIR),

    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=WARMUP_STEPS,
    max_grad_norm=1.0,

    num_train_epochs=NUM_TRAIN_EPOCHS,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,

    gradient_checkpointing=True,

    optim="adafactor",
    torch_empty_cache_steps=1,

    bf16=USE_BF16,
    fp16=USE_FP16,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    save_total_limit=2,
    predict_with_generate=False,

    report_to="none",
    disable_tqdm=True,

    seed=SEED,
    data_seed=SEED,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_validation,
    data_collator=data_collator,
    processing_class=tokenizer,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOPPING_PATIENCE
        )
    ],
)

logger.info(
    "Koraci optimizacije/epoha=%d, ukupno=%d, warmup=%d",
    optimizer_steps_per_epoch,
    TOTAL_OPTIMIZER_STEPS,
    WARMUP_STEPS,
)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.33GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.33GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

INFO:mt5_gold:Koraci optimizacije/epoha=13, ukupno=104, warmup=10


### Fine-tune mT5-Gold
Only runs real training

In [15]:
train_result = trainer.train()

logger.info(
    "Training finished. Best checkpoint: %s",
    trainer.state.best_model_checkpoint
)

trainer.save_model(
    str(BEST_MODEL_DIR)
)

tokenizer.save_pretrained(
    str(BEST_MODEL_DIR)
)

training_history_df = pd.DataFrame(
    trainer.state.log_history
)

training_history_df.to_csv(
    ARTIFACTS_DIR / "training_history.csv",
    index=False
)

train_metrics = {
    key: (
        float(value)
        if isinstance(value, (int, float, np.number))
        else value
    )
    for key, value in train_result.metrics.items()
}

with (ARTIFACTS_DIR / "train_metrics.json").open(
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        train_metrics,
        file,
        ensure_ascii=False,
        indent=2
    )

final_eval_metrics = trainer.evaluate()

with (ARTIFACTS_DIR / "validation_loss_metrics.json").open(
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        {
            key: float(value)
            if isinstance(value, (int, float, np.number))
            else value
            for key, value in final_eval_metrics.items()
        },
        file,
        ensure_ascii=False,
        indent=2
    )

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


{'loss': '110.9', 'grad_norm': '1.748e+04', 'learning_rate': '9.787e-05', 'epoch': '1'}
{'eval_loss': '8.212', 'eval_runtime': '3.236', 'eval_samples_per_second': '6.49', 'eval_steps_per_second': '6.49', 'epoch': '1'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '83.56', 'grad_norm': '6967', 'learning_rate': '8.404e-05', 'epoch': '2'}
{'eval_loss': '6.92', 'eval_runtime': '5.351', 'eval_samples_per_second': '3.925', 'eval_steps_per_second': '3.925', 'epoch': '2'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '872.5', 'train_samples_per_second': '0.917', 'train_steps_per_second': '0.119', 'train_loss': '55.45', 'epoch': '8'}


[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].
INFO:mt5_gold:Training finished. Best checkpoint: /content/drive/MyDrive/Student-QA/artifacts/seq2seq/mt5_gold_v2_fp32/checkpoints/checkpoint-104


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': '3.05', 'eval_runtime': '3.807', 'eval_samples_per_second': '5.516', 'eval_steps_per_second': '5.516', 'epoch': '8'}


In [23]:
print("Trainer best:")
print(trainer.state.best_model_checkpoint)

print("\nMODEL_LOAD_DIR:")
print(str(MODEL_LOAD_DIR))

print(
    "\nSame:",
    str(MODEL_LOAD_DIR) == trainer.state.best_model_checkpoint
)

Trainer best:
/content/drive/MyDrive/Student-QA/artifacts/seq2seq/mt5_gold_v2_fp32/checkpoints/checkpoint-104

MODEL_LOAD_DIR:
/content/drive/MyDrive/Student-QA/artifacts/seq2seq/mt5_gold_v2_fp32/checkpoints/checkpoint-104

Same: True


In [18]:
print("BEST CHECKPOINT:")
print(trainer.state.best_model_checkpoint)

print("\nBEST METRIC:")
print(trainer.state.best_metric)

BEST CHECKPOINT:
/content/drive/MyDrive/Student-QA/artifacts/seq2seq/mt5_gold_v2_fp32/checkpoints/checkpoint-104

BEST METRIC:
3.05012583732605


In [24]:
experiment_config = {
    "experiment": EXPERIMENT_NAME,
    "model_name": MODEL_NAME,
    "training_context": "gold",
    "generator_document_field": "processed_text",
    "generator_question_field": "processed_question",
    "target_field": "processed_answer",
    "seed": SEED,
    "max_input_length": MAX_INPUT_LENGTH,
    "max_target_length": MAX_TARGET_LENGTH,
    "rag_top_k": RAG_TOP_K,
    "max_gold_context_chunks": MAX_GOLD_CONTEXT_CHUNKS,
    "learning_rate": LEARNING_RATE,
    "num_train_epochs": NUM_TRAIN_EPOCHS,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "weight_decay": WEIGHT_DECAY,
    "warmup_steps": WARMUP_STEPS,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "best_checkpoint": trainer.state.best_model_checkpoint,
    "n_train": len(train_gold),
    "n_validation": len(validation_gold),
}

with (ARTIFACTS_DIR / "experiment_config.json").open(
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        experiment_config,
        file,
        ensure_ascii=False,
        indent=2
    )

logger.info(
    "Model, tokenizer, metrike and konfiguracija sacuvani u %s",
    ARTIFACTS_DIR
)

INFO:mt5_gold:Model, tokenizer, metrike and konfiguracija sacuvani u /content/drive/MyDrive/Student-QA/artifacts/seq2seq/mt5_gold_v2_fp32


In [25]:
# Provera generacije na validation primerima sa GOLD kontekstom

model_for_check = trainer.model
model_for_check.eval()

for i in range(min(5, len(validation_gold))):
    example = validation_gold[i]

    encoded = tokenizer(
        example["input_text"],
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_LENGTH,
    ).to(DEVICE)

    with torch.no_grad():
        generated_ids = model_for_check.generate(
            **encoded,
            max_new_tokens=MAX_TARGET_LENGTH,
            num_beams=4,
            bad_words_ids=[
                [token_id]
                for token_id in sentinel_token_ids
            ],
        )

    prediction = tokenizer.decode(
        generated_ids[0],
        skip_special_tokens=True,
    )

    print("=" * 100)
    print(f"PRIMER {i + 1}")

    print("\nREFERENTNI ODGOVOR:")
    print(example["target_text"])

    print("\nGENERISANI ODGOVOR:")
    print(prediction)

PRIMER 1

REFERENTNI ODGOVOR:
Cachegrind je Valgrind alat za profilisanje keš memorije. Koristi se pokretanjem programa kroz Cachegrind, nakon čega se analiziraju prikupljene informacije o pristupima kešu i izvršavanju.

GENERISANI ODGOVOR:
<extra_id_0> se mogu rekonstruisati i profili grana. Slika 6.7: Potpuno dupliranje koda radi periodičnog izvršavanja instrumentovanog koda odnosu na profajliranje na osnovu fiksiranih vremenskih intervala.
PRIMER 2

REFERENTNI ODGOVOR:
Instrumentaciono profajliranje koristi dodatno ubačen kod za prikupljanje tačnih podataka o izvršavanju, na primer broja poziva funkcija ili izvršenih grana i blokova.

GENERISANI ODGOVOR:
<extra_id_0> izvršavanja programa.
PRIMER 3

REFERENTNI ODGOVOR:
Dobar skup testova treba da efikasno otkriva greške, bude relativno mali i brz za izvršavanje i da pruža visok stepen poverenja u pouzdanost softvera.

GENERISANI ODGOVOR:
<extra_id_0> i da se provera da li je softver spreman za rad u stvarnom okruženju korisnika.
PRIM

### RAG inference with the saved mT5-Gold model

From now on generator is **fixed**. Only retrieved context is changed:

`data/retrieval/{retriever}_{split}_top10.jsonl`

and every record has `retrieved_chunks`, and every chunk has`processed_text`


In [ ]:
# print("BEST_MODEL_DIR:")
# for p in BEST_MODEL_DIR.iterdir():
#     print(" ", p.name)

# print("\nCHECKPOINTS:")
# for checkpoint in sorted(CHECKPOINT_DIR.glob("checkpoint-*")):
#     print("\n", checkpoint)
#     for p in checkpoint.iterdir():
#         print(" ", p.name)

In [ ]:
# MODEL_LOAD_DIR = CHECKPOINT_DIR / "checkpoint-65"

# inference_tokenizer = AutoTokenizer.from_pretrained(
#     MODEL_LOAD_DIR
# )

# inference_model = AutoModelForSeq2SeqLM.from_pretrained(
#     MODEL_LOAD_DIR
# ).to(DEVICE)

# inference_model.config.use_cache = True
# inference_model.eval()

In [ ]:
# inference_model.save_pretrained(
#     BEST_MODEL_DIR,
#     safe_serialization=True
# )

# inference_tokenizer.save_pretrained(
#     BEST_MODEL_DIR
# )

In [26]:
inference_tokenizer = AutoTokenizer.from_pretrained(
    BEST_MODEL_DIR
)

inference_model = AutoModelForSeq2SeqLM.from_pretrained(
    BEST_MODEL_DIR
).to(DEVICE)

inference_model.config.use_cache = True
inference_model.eval()


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


MT5ForConditionalGeneration(
  (shared): Embedding(250112, 768)
  (encoder): MT5Stack(
    (embed_tokens): Embedding(250112, 768)
    (block): ModuleList(
      (0): MT5Block(
        (layer): ModuleList(
          (0): MT5LayerSelfAttention(
            (SelfAttention): MT5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): MT5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): MT5LayerFF(
            (DenseReluDense): MT5DenseGatedActDense(
              (wi_0): Linear(in_features=768, out_features=2048, bias=False)
              (wi_1): Linear(in_features=768, out_features=2048, bias=False)
         

### Build context from retrieval results

In [27]:
def build_rag_input(
    retrieval_record: dict,
    top_k: int = RAG_TOP_K
) -> str:

    retrieved_chunks = retrieval_record.get(
        "retrieved_chunks",
        []
    )[:top_k]

    question = retrieval_record.get(
        "processed_question",
        retrieval_record.get("question", "")
    )

    context, _, _ = pack_context_to_input_budget(
        question,
        retrieved_chunks,
        max_input_length=MAX_INPUT_LENGTH
    )

    return build_model_input(
        question,
        context
    )

### Batched generation

In [28]:
@torch.inference_mode()
def generate_batch(
    input_texts: list[str]
) -> list[str]:

    encoded = inference_tokenizer(
        input_texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_INPUT_LENGTH,
    ).to(DEVICE)

    generated_ids = inference_model.generate(
        **encoded,
        max_new_tokens=GENERATION_MAX_NEW_TOKENS,
        num_beams=GENERATION_NUM_BEAMS,
        early_stopping=True,
        no_repeat_ngram_size=3,
        bad_words_ids=[
            [token_id]
            for token_id in sentinel_token_ids
        ],
    )

    return inference_tokenizer.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )


### Generate and save one RAG experiment

In [29]:
def run_rag_inference(
    retriever_name: str,
    split_name: str,
    top_k: int = RAG_TOP_K,
    batch_size: int = GENERATION_BATCH_SIZE,
) -> Path:

    retrieval_path = (
        RETRIEVAL_DIR
        / f"{retriever_name}_{split_name}_top{TOP_K_EXPORTED}.jsonl"
    )

    if not retrieval_path.exists():
        raise FileNotFoundError(
            f"Nedostaje retrieval fajl: {retrieval_path}"
        )

    retrieval_records = load_jsonl(
        retrieval_path
    )

    outputs = []

    for start in range(
        0,
        len(retrieval_records),
        batch_size
    ):
        batch_records = retrieval_records[
            start:start + batch_size
        ]

        batch_inputs = [
            build_rag_input(
                record,
                top_k=top_k
            )
            for record in batch_records
        ]

        generated_answers = generate_batch(
            batch_inputs
        )

        for record, prediction in zip(
            batch_records,
            generated_answers
        ):
            outputs.append({
                "question_id": record["question_id"],
                "question": record["question"],
                "gold_answer": record["answer"],
                "source_pages": record["source_pages"],
                "retriever": retriever_name,
                "generator": "mt5_gold",
                "rag_top_k": top_k,
                "generated_answer": prediction.strip(),
                "retrieved_chunks": record["retrieved_chunks"][:top_k],
            })

    output_path = (
        GENERATION_DIR
        / f"{retriever_name}_{split_name}_top{top_k}_predictions.jsonl"
    )

    save_jsonl(
        outputs,
        output_path
    )

    logger.info(
        "Generated %d predictions: %s",
        len(outputs),
        output_path
    )

    return output_path

### Validation inference: 3 mT5-Gold RAG experiments

We make 3 separate prediction files (for evaluation later)


In [30]:
RETRIEVERS = (
    "tfidf",
    "bm25",
    "dense",
)

validation_prediction_paths = {}

for retriever_name in RETRIEVERS:
    expected_path = (
        RETRIEVAL_DIR
        / f"{retriever_name}_validation_top{TOP_K_EXPORTED}.jsonl"
    )

    if not expected_path.exists():
        logger.warning(
            "Validation inference skipped for %s: missing %s",
            retriever_name,
            expected_path
        )
        continue

    validation_prediction_paths[
        retriever_name
    ] = run_rag_inference(
        retriever_name=retriever_name,
        split_name="validation",
        top_k=RAG_TOP_K,
    )

with (ARTIFACTS_DIR / "validation_prediction_files.json").open(
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        {
            name: str(path)
            for name, path
            in validation_prediction_paths.items()
        },
        file,
        ensure_ascii=False,
        indent=2
    )


INFO:mt5_gold:Generated 21 predictions: /content/drive/MyDrive/Student-QA/artifacts/seq2seq/mt5_gold_v2_fp32/generation/tfidf_validation_top5_predictions.jsonl
INFO:mt5_gold:Generated 21 predictions: /content/drive/MyDrive/Student-QA/artifacts/seq2seq/mt5_gold_v2_fp32/generation/bm25_validation_top5_predictions.jsonl
INFO:mt5_gold:Generated 21 predictions: /content/drive/MyDrive/Student-QA/artifacts/seq2seq/mt5_gold_v2_fp32/generation/dense_validation_top5_predictions.jsonl


### Final test inference — locked by default

`RUN_FINAL_TEST` stays `False` till:
- mT5-Gold checkpoint is not locked,
- `RAG_TOP_K` is not locked,
- all 3 retrievers are not locked,
- validation set decisions are not made

In [31]:
RUN_FINAL_TEST = False

if RUN_FINAL_TEST:
    test_prediction_paths = {}

    for retriever_name in RETRIEVERS:
        test_prediction_paths[
            retriever_name
        ] = run_rag_inference(
            retriever_name=retriever_name,
            split_name="test",
            top_k=RAG_TOP_K,
        )

    with (ARTIFACTS_DIR / "test_prediction_files.json").open(
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(
            {
                name: str(path)
                for name, path
                in test_prediction_paths.items()
            },
            file,
            ensure_ascii=False,
            indent=2
        )


## Outputs

### Saved model
- `artifacts/seq2seq/mt5_gold/best_model/`
  - model weights
  - tokenizer
  - model configuration

### Training / reproducibility artifacts
- `split_audit.json`
- `input_length_stats.json`
- `training_history.csv`
- `train_metrics.json`
- `validation_loss_metrics.json`
- `experiment_config.json`
- `training.log`
- prepared Gold train/validation datasets

### Validation RAG predictions
- `data/generation/mt5_gold/tfidf_validation_top5_predictions.jsonl`
- `data/generation/mt5_gold/bm25_validation_top5_predictions.jsonl`
- `data/generation/mt5_gold/dense_validation_top5_predictions.jsonl`

### Final test predictions
 `RUN_FINAL_TEST = True`

In [32]:
# Save reproducible RAG configuration manifests

RAG_CONFIG_DIR = ARTIFACTS_DIR / "rag_configs"
RAG_CONFIG_DIR.mkdir(
    parents=True,
    exist_ok=True
)

GENERATOR_MODEL_DIR = BEST_MODEL_DIR
if not (
    (GENERATOR_MODEL_DIR / "model.safetensors").exists()
    or
    (GENERATOR_MODEL_DIR / "pytorch_model.bin").exists()
):
    raise FileNotFoundError(
        f"Generator weights nisu pronađeni u {GENERATOR_MODEL_DIR}"
    )


for retriever_name in RETRIEVERS:

    retrieval_path = (
        RETRIEVAL_DIR
        / f"{retriever_name}_validation_top{TOP_K_EXPORTED}.jsonl"
    )

    prediction_path = (
        GENERATION_DIR
        / f"{retriever_name}_validation_top{RAG_TOP_K}_predictions.jsonl"
    )

    rag_config = {
        "rag_name": (
            f"{retriever_name}_{EXPERIMENT_NAME}"
        ),

        "retriever": retriever_name,

        "retrieval_input": {
            "file": str(retrieval_path),
            "exported_top_k": TOP_K_EXPORTED,
            "rag_top_k": RAG_TOP_K,
        },

        "generator": {
            "experiment": EXPERIMENT_NAME,
            "base_model": MODEL_NAME,
            "checkpoint": str(GENERATOR_MODEL_DIR),
            "max_input_length": MAX_INPUT_LENGTH,
            "max_target_length": MAX_TARGET_LENGTH,
            "training_context": "gold_token_budgeted",
        },

        "generation": {
            "num_beams": GENERATION_NUM_BEAMS,
            "max_new_tokens": GENERATION_MAX_NEW_TOKENS,
        },

        "validation_output": (
            str(prediction_path)
            if prediction_path.exists()
            else None
        ),
    }

    output_path = (
        RAG_CONFIG_DIR
        / f"{retriever_name}_{EXPERIMENT_NAME}.json"
    )

    with output_path.open(
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(
            rag_config,
            file,
            ensure_ascii=False,
            indent=2
        )

    logger.info(
        "RAG config saved: %s",
        output_path
    )

INFO:mt5_gold:RAG config saved: /content/drive/MyDrive/Student-QA/artifacts/seq2seq/mt5_gold_v2_fp32/rag_configs/tfidf_mt5_gold_v2_fp32.json
INFO:mt5_gold:RAG config saved: /content/drive/MyDrive/Student-QA/artifacts/seq2seq/mt5_gold_v2_fp32/rag_configs/bm25_mt5_gold_v2_fp32.json
INFO:mt5_gold:RAG config saved: /content/drive/MyDrive/Student-QA/artifacts/seq2seq/mt5_gold_v2_fp32/rag_configs/dense_mt5_gold_v2_fp32.json
